# 01 DLH713 Trajectory Data Preparation

Goal: prepare the current DLH713 aircraft trajectory exports for later PCA, HDBSCAN clustering, visualization, and anomaly analysis.

This notebook follows a readable data-preparation workflow:

- define dataset paths;
- verify that raw files exist;
- inspect the raw export structure;
- run the reusable preprocessing pipeline from `src/preprocess.py`;
- load and validate the generated outputs;
- explain what should be passed to the next teammates.

No PCA, HDBSCAN, clustering, or model training is performed here.


In [ ]:
from pathlib import Path
import sys
import json

import pandas as pd


## 1. Dataset Paths

Define the project root, raw-data location, filename pattern, and output directory.

The current repository stores raw DLH713 files in the project root. If the team later moves them into `data/raw/`, only `RAW_DIR` needs to be changed.


In [ ]:
PROJECT_ROOT = Path.cwd().resolve()

# Current setup: raw files are in the repository root.
RAW_DIR = PROJECT_ROOT

# Change this if the team later uses another callsign or filename pattern.
FILE_PATTERN = "DLH713_*.xlsx"

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Raw data directory:", RAW_DIR)
print("File pattern:", FILE_PATTERN)
print("Processed data directory:", PROCESSED_DIR)


## 2. Verify Raw Dataset Files

Before preprocessing, confirm which raw files will be used.


In [ ]:
raw_files = sorted(RAW_DIR.glob(FILE_PATTERN))

for path in raw_files:
    size_mb = path.stat().st_size / (1024 ** 2)
    print(f"✓ {path.name:<30} {size_mb:,.2f} MB")

if not raw_files:
    raise FileNotFoundError(
        f"No files found in {RAW_DIR} using pattern {FILE_PATTERN!r}"
    )

print(f"\nTotal raw files found: {len(raw_files)}")


## 3. Inspect Raw File Structure

The raw files are Excel workbooks, but their content is not a normal table. Each row contains CSV-like text fragments. Long trajectory records may be split across several rows.

This is why preprocessing needs a reconstruction step before normal cleaning.


In [ ]:
sample_raw = pd.read_excel(raw_files[0], header=None, dtype=str, engine="openpyxl")

print("Sample raw file:", raw_files[0].name)
print("Shape:", sample_raw.shape)
sample_raw.head(10)


## 4. Load the Reusable Preprocessing Code

The heavy preprocessing logic is kept in `src/preprocess.py` so that it is reproducible from both this notebook and the command line. This avoids copying a long implementation into multiple notebooks.


In [ ]:
src_path = PROJECT_ROOT / "src"
if str(src_path) not in sys.path:
    sys.path.append(str(src_path))

from preprocess import run_preprocessing

print("Imported run_preprocessing from:", src_path / "preprocess.py")


## 5. Run Preprocessing

This step reconstructs split flight records, expands nested `track` lists into point-level rows, removes invalid points and duplicates, sorts trajectories by time, and creates standardized columns for later PCA/HDBSCAN work.


In [ ]:
report = run_preprocessing(
    raw_dir=RAW_DIR,
    file_pattern=FILE_PATTERN,
    processed_dir=PROCESSED_DIR,
)

report_dict = report.__dict__
print(json.dumps(report_dict, indent=2))


## 6. Load Generated Outputs

The preprocessing pipeline creates four files: `flight_points_clean.csv`, `flight_points_scaled.csv`, `flights_summary.csv`, and `preprocessing_report.json`.


In [ ]:
clean_path = PROCESSED_DIR / "flight_points_clean.csv"
scaled_path = PROCESSED_DIR / "flight_points_scaled.csv"
summary_path = PROCESSED_DIR / "flights_summary.csv"
report_path = PROCESSED_DIR / "preprocessing_report.json"

clean_points = pd.read_csv(clean_path)
scaled_points = pd.read_csv(scaled_path)
flights_summary = pd.read_csv(summary_path)

print("Clean points:", clean_points.shape)
print("Scaled points:", scaled_points.shape)
print("Flights summary:", flights_summary.shape)


## 7. Preview Clean Point-Level Data

Each row represents one trajectory point from one flight.


In [ ]:
clean_points.head()


## 8. Validate Data Quality

Check that coordinates, heading values, and timestamps are valid after preprocessing.


In [ ]:
quality_checks = {
    "rows": len(clean_points),
    "unique_flights": clean_points["flight_id"].nunique(),
    "missing_latitude": int(clean_points["latitude"].isna().sum()),
    "missing_longitude": int(clean_points["longitude"].isna().sum()),
    "invalid_latitude": int((~clean_points["latitude"].between(-90, 90)).sum()),
    "invalid_longitude": int((~clean_points["longitude"].between(-180, 180)).sum()),
    "invalid_heading": int((~clean_points["heading"].between(0, 360)).sum()),
}

quality_checks


## 9. Scaled Dataset for Next Teammates

The scaled dataset keeps original columns and adds standardized versions of selected numeric columns. The next teammates can use this file for PCA, HDBSCAN, or other feature-based experiments.


In [ ]:
scaled_columns = [column for column in scaled_points.columns if column.endswith("_scaled")]
print("Scaled columns:")
for column in scaled_columns:
    print("-", column)

scaled_points[["flight_id"] + scaled_columns].head()


## 10. Final Handoff

My preprocessing contribution ends with these files:

```text
data/processed/flight_points_clean.csv
data/processed/flight_points_scaled.csv
data/processed/flights_summary.csv
data/processed/preprocessing_report.json
```

Recommended usage for teammates:

- use `flight_points_clean.csv` for map visualization and trajectory inspection;
- use `flight_points_scaled.csv` for PCA/HDBSCAN experiments;
- use `flights_summary.csv` for flight-level quality checks;
- use `preprocessing_report.json` to reproduce and report preprocessing statistics.

The exact original data provider and route labels should still be confirmed by the team. The preprocessing pipeline only depends on the observed file structure and trajectory fields, not on airport names.
